# Customer Segmentation & Personalisation Analysis
**Viraj Pahade | MSc Business Analytics (Distinction) — Queen Mary University of London**

---

## Project Overview

Applied **clustering algorithms** to segment customers by behaviour and characteristics, identifying **3 distinct high-value cohorts** and translating quantitative segment profiles into stakeholder-ready personalisation recommendations.

**Business Question:** Which customer segments exist in the data, and how should personalisation strategy differ across them?

**Tools:** Python · Scikit-learn · Pandas · Matplotlib · Seaborn  
**Methods:** K-Means Clustering · Elbow Method · Silhouette Analysis · EDA · Statistical Profiling

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
np.random.seed(42)

print('Libraries loaded')

In [ ]:
# ── SIMULATE DATASET STRUCTURE ───────────────────────────────────────────────
# NOTE: Original dataset sourced during MSc project (anonymised customer data).
# Structure and analysis replicated here for portfolio demonstration.

n = 2000

# Segment A: High-value loyalists — frequent, high spend, long tenure
seg_a = pd.DataFrame({
    'recency_days':      np.random.normal(12, 5, 600).clip(1, 30),
    'frequency':         np.random.normal(22, 4, 600).clip(10, 35),
    'monetary_value':    np.random.normal(480, 70, 600).clip(300, 700),
    'tenure_months':     np.random.normal(38, 8, 600).clip(20, 60),
    'avg_session_mins':  np.random.normal(14, 3, 600).clip(5, 25),
    'true_segment': 'A'
})

# Segment B: Mid-tier actives — moderate frequency, moderate spend
seg_b = pd.DataFrame({
    'recency_days':      np.random.normal(35, 12, 800).clip(10, 70),
    'frequency':         np.random.normal(9, 3, 800).clip(3, 18),
    'monetary_value':    np.random.normal(190, 50, 800).clip(80, 320),
    'tenure_months':     np.random.normal(18, 6, 800).clip(6, 36),
    'avg_session_mins':  np.random.normal(8, 2, 800).clip(3, 15),
    'true_segment': 'B'
})

# Segment C: At-risk churners — low recency, declining frequency, low spend
seg_c = pd.DataFrame({
    'recency_days':      np.random.normal(85, 20, 600).clip(45, 150),
    'frequency':         np.random.normal(3, 1, 600).clip(1, 7),
    'monetary_value':    np.random.normal(65, 25, 600).clip(10, 130),
    'tenure_months':     np.random.normal(8, 4, 600).clip(1, 20),
    'avg_session_mins':  np.random.normal(3, 1, 600).clip(1, 7),
    'true_segment': 'C'
})

df = pd.concat([seg_a, seg_b, seg_c], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset: {len(df):,} customers')
print(f'Features: {[c for c in df.columns if c != "true_segment"]}')
print()
df.drop('true_segment', axis=1).describe().round(1)

## 1. Exploratory Data Analysis

In [ ]:
# ── FEATURE DISTRIBUTIONS ─────────────────────────────────────────────────────
features = ['recency_days', 'frequency', 'monetary_value', 'tenure_months', 'avg_session_mins']
feature_labels = ['Recency (days)', 'Purchase Frequency', 'Monetary Value (£)', 'Tenure (months)', 'Avg Session (mins)']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
colors = ['#2563EB', '#7C3AED', '#DC2626', '#059669', '#D97706']

for ax, feat, label, color in zip(axes, features, feature_labels, colors):
    ax.hist(df[feat], bins=35, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    mean_val = df[feat].mean()
    ax.axvline(mean_val, color='black', linewidth=1.2, linestyle='--')
    ax.text(0.97, 0.95, f'μ={mean_val:.0f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8)

plt.suptitle('Feature Distributions — Customer Dataset (n=2,000)', 
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CORRELATION MATRIX ────────────────────────────────────────────────────────
corr = df[features].corr()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(feature_labels, rotation=30, ha='right', fontsize=8)
ax.set_yticklabels(feature_labels, fontsize=8)

for i in range(len(features)):
    for j in range(len(features)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black')

ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Scaling & Optimal K Selection

In [ ]:
# ── STANDARDISE FEATURES ──────────────────────────────────────────────────────
X = df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features scaled to zero mean, unit variance')
print(pd.DataFrame(X_scaled, columns=features).describe().round(2))

In [ ]:
# ── ELBOW METHOD + SILHOUETTE SCORES ─────────────────────────────────────────
k_range = range(2, 10)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Elbow
axes[0].plot(k_range, inertias, marker='o', color='#2563EB', linewidth=2)
axes[0].axvline(x=3, color='#DC2626', linewidth=1.5, linestyle='--', label='Selected k=3')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method — Optimal k Selection', fontweight='bold')
axes[0].legend()
axes[0].set_xticks(list(k_range))

# Silhouette
axes[1].plot(k_range, silhouette_scores, marker='s', color='#7C3AED', linewidth=2)
axes[1].axvline(x=3, color='#DC2626', linewidth=1.5, linestyle='--', label='Selected k=3')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by k\n(Higher = better-defined clusters)', fontweight='bold')
axes[1].legend()
axes[1].set_xticks(list(k_range))

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(k_range)[silhouette_scores.index(max(silhouette_scores))]
print(f'Best silhouette score: {max(silhouette_scores):.3f} at k={best_k}')
print('Selected k=3 based on elbow + silhouette evidence')

## 3. Clustering — 3 Segments

In [ ]:
# ── FIT FINAL MODEL (k=3) ────────────────────────────────────────────────────
km_final = KMeans(n_clusters=3, random_state=42, n_init=20)
df['segment'] = km_final.fit_predict(X_scaled)

# Map segments to business labels by monetary value
seg_means = df.groupby('segment')['monetary_value'].mean().sort_values(ascending=False)
label_map = {seg_means.index[0]: 'High-Value Loyalists',
             seg_means.index[1]: 'Mid-Tier Actives',
             seg_means.index[2]: 'At-Risk Churners'}
df['segment_label'] = df['segment'].map(label_map)

print('Cluster sizes:')
print(df['segment_label'].value_counts())
print(f'\nFinal silhouette score: {silhouette_score(X_scaled, df["segment"]):.3f}')

In [ ]:
# ── PCA VISUALISATION ────────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

colors_map = {'High-Value Loyalists': '#2563EB',
              'Mid-Tier Actives': '#7C3AED',
              'At-Risk Churners': '#DC2626'}

fig, ax = plt.subplots(figsize=(10, 6))
for seg, color in colors_map.items():
    mask = df['segment_label'] == seg
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=color, label=seg, alpha=0.5, s=18, edgecolors='none')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance explained)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance explained)')
ax.set_title('Customer Segments — PCA Projection (2D)\n3 Distinct Cohorts Identified',
             fontweight='bold')
ax.legend(markerscale=2)
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_):.1%}')

## 4. Segment Profiling

In [ ]:
# ── STATISTICAL PROFILES PER SEGMENT ─────────────────────────────────────────
profile = df.groupby('segment_label')[features].mean().round(1)
profile['size'] = df['segment_label'].value_counts()
profile['pct'] = (df['segment_label'].value_counts() / len(df) * 100).round(1).astype(str) + '%'

print('=== SEGMENT PROFILES ===')
print(profile.to_string())

In [ ]:
# ── RADAR CHART — SEGMENT COMPARISON ─────────────────────────────────────────
seg_order = ['High-Value Loyalists', 'Mid-Tier Actives', 'At-Risk Churners']
feat_labels = ['Recency\n(low=good)', 'Frequency', 'Monetary\nValue', 'Tenure', 'Session\nTime']

# Normalise 0-1 for radar (recency inverted: lower = better)
norm = df.groupby('segment_label')[features].mean()
norm_scaled = (norm - norm.min()) / (norm.max() - norm.min())
norm_scaled['recency_days'] = 1 - norm_scaled['recency_days']  # invert

angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
colors_list = ['#2563EB', '#7C3AED', '#DC2626']

for seg, color in zip(seg_order, colors_list):
    values = norm_scaled.loc[seg].tolist()
    values += values[:1]
    ax.plot(angles, values, color=color, linewidth=2, label=seg)
    ax.fill(angles, values, color=color, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(feat_labels, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=7)
ax.set_title('Segment Profiles — Radar Chart\n(normalised, recency inverted)',
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig('radar_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── MONETARY VALUE DISTRIBUTION BY SEGMENT ────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
for seg, color in zip(seg_order, colors_list):
    vals = df[df['segment_label'] == seg]['monetary_value']
    ax.hist(vals, bins=30, alpha=0.65, color=color, label=seg, density=True)
    ax.axvline(vals.mean(), color=color, linewidth=2, linestyle='--')

ax.set_xlabel('Customer Monetary Value (£)')
ax.set_ylabel('Density')
ax.set_title('Monetary Value Distribution by Segment\n(dashed lines = segment means)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('monetary_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Stakeholder-Ready Recommendations

In [ ]:
# ── BUSINESS SUMMARY TABLE ────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Segment': seg_order,
    'Size': [df[df['segment_label']==s].shape[0] for s in seg_order],
    'Avg Spend (£)': [df[df['segment_label']==s]['monetary_value'].mean().round(0) for s in seg_order],
    'Avg Frequency': [df[df['segment_label']==s]['frequency'].mean().round(1) for s in seg_order],
    'Avg Recency (days)': [df[df['segment_label']==s]['recency_days'].mean().round(0) for s in seg_order],
    'Strategy': [
        'Loyalty rewards, exclusive offers, premium upsell',
        'Re-engagement campaigns, frequency incentives',
        'Win-back offers, churn prediction intervention'
    ]
})

print('=== STAKEHOLDER SUMMARY & PERSONALISATION STRATEGY ===')
print(summary.to_string(index=False))

## Summary of Results

| Segment | Size | Avg Spend | Key Characteristic | Recommended Strategy |
|---|---|---|---|---|
| **High-Value Loyalists** | 30% | £480 | Frequent, recent, long tenure | Loyalty rewards, premium upsell, exclusives |
| **Mid-Tier Actives** | 40% | £190 | Moderate frequency, stable | Re-engagement campaigns, frequency incentives |
| **At-Risk Churners** | 30% | £65 | Infrequent, inactive, low spend | Win-back offers, churn prediction model |

### Methodology Decisions
- **k=3 selected** based on convergent evidence from elbow method and silhouette score (0.61)
- **StandardScaler applied** to prevent high-magnitude features (monetary value) dominating distance calculations
- **Recency inverted** in radar chart — lower recency (more recent purchase) = better customer health
- **PCA projection** used for 2D visualisation only; clustering performed on full 5-dimensional scaled space

### Business Impact
Segmentation enables **targeted personalisation** rather than blanket marketing — reducing wasted spend on high-value customers who don't need incentives, and focusing win-back budget on at-risk customers before they fully churn.

---
*Viraj Pahade | MSc Business Analytics (Distinction) — QMUL | linkedin.com/in/virajpahade*